# Prepare dspy dataset

In [52]:
input_dir = "../output/dspy-extract"
input_reviewed = "../output/dataset/reviewed.jsonl"
output_dataset = "../output/dataset/dspy-dataset.csv"

## Load extracted

Load extracted pages and merge with reviews.

In [53]:
from glob import glob
from pathlib import Path

input_files = [
    Path(x)
    for x in sorted(glob(f"{input_dir}/*/*.json"))
]
input_files[:5]

[PosixPath('../output/dspy-extract/pkna-0-2/PK0-2 002.json'),
 PosixPath('../output/dspy-extract/pkna-0-2/PK0-2 003.json'),
 PosixPath('../output/dspy-extract/pkna-0-2/PK0-2 004.json'),
 PosixPath('../output/dspy-extract/pkna-0-2/PK0-2 005.json'),
 PosixPath('../output/dspy-extract/pkna-0-2/PK0-2 006.json')]

In [54]:
import json
from dataclasses import dataclass

@dataclass
class ExtractedLine:
    character: str
    text: str

@dataclass
class ExtractedPage:
    input_path: Path
    uno_present: bool
    dialogue: list[ExtractedLine]
    page_num: int = 0
    
    @property
    def issue(self) -> str:
        pkna = self.input_path.parent.name
        num_str = pkna.strip("pkna-").replace("-", "/")
        return f"PKNA #{num_str}"


def load_extracted_page(file_path: Path) -> ExtractedPage:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return ExtractedPage(
        input_path=Path(data["input_path"]),
        uno_present=data["extracted"]["uno_present"],
        dialogue=[
            ExtractedLine(character=line["character"], text=line["text"])
            for line in data["extracted"]["dialogue"]
        ],
    )

extracted_pages = [load_extracted_page(file) for file in input_files]

In [55]:
# Load reviewed pages
with open(input_reviewed, "r", encoding="utf-8") as f:
    reviewed_data = [json.loads(line) for line in f]
    review_map = {
        Path(item["image"]): item
        for item in reviewed_data
    }

def parse_character(character: str) -> str:
    match character.lower():
        case "pk" | "pkna" | "paperinik":
            return "pk"
        case _:
            return character.lower()

for page in extracted_pages:
    if page.input_path in review_map:
        review = review_map[page.input_path]
        page.dialogue = [
            ExtractedLine(character=parse_character(line["character"]), text=line["text"])
            for line in review["ocr"]["dialogue"]
        ]
        page.uno_present = any(
            line.character == "uno" for line in page.dialogue
        )

len(extracted_pages), len(review_map)

(3316, 62)

In [56]:
from collections import defaultdict

def compute_page_nums(paths: list[Path]) -> dict[str, int]:
    # Map directory names to their image files
    per_issue: dict[str, list[str]] = defaultdict(list)
    for file in paths:
        dir_name = file.parent.name
        per_issue[dir_name].append(str(file))

    # Compute page numbers for each issue
    page_nums: dict[str, int] = {}
    for _, images in per_issue.items():
        for i, image in enumerate(images):
            page_nums[image] = i + 1  # Page numbers start from 1

    return page_nums

page_nums = compute_page_nums([
    page.input_path for page in extracted_pages
])

for page in extracted_pages:
    page.page_num = page_nums[str(page.input_path)]


def issue_sort_key(issue: str) -> float:
    """Sort issues by their numeric part."""
    num_str = issue.strip("PKNA #").replace("/", ".")
    return float(num_str)

extracted_pages.sort(key=lambda x: (issue_sort_key(x.issue), x.page_num))
extracted_pages[:5]

[ExtractedPage(input_path=PosixPath('../input/pkna/pkna-0/pkna-0-000.jpg'), uno_present=False, dialogue=[], page_num=1),
 ExtractedPage(input_path=PosixPath('../input/pkna/pkna-0/pkna-0-001.jpg'), uno_present=False, dialogue=[], page_num=2),
 ExtractedPage(input_path=PosixPath('../input/pkna/pkna-0/pkna-0-002.jpg'), uno_present=False, dialogue=[], page_num=3),
 ExtractedPage(input_path=PosixPath('../input/pkna/pkna-0/pkna-0-003.jpg'), uno_present=False, dialogue=[], page_num=4),
 ExtractedPage(input_path=PosixPath('../input/pkna/pkna-0/pkna-0-004.jpg'), uno_present=False, dialogue=[], page_num=5)]

In [57]:
# Filter out pages without dialogue
extracted_pages = [
    page for page in extracted_pages
    if page.dialogue
]
len(extracted_pages)

436

## Merge contiguous pages

In [58]:
@dataclass
class Conversation:
    pages: list[ExtractedPage]

    @property
    def issue(self) -> str:
        """Get the issue number of the first page in the conversation."""
        return self.pages[0].issue if self.pages else ''

    @property
    def characters(self) -> list[str]:
        """Get the set of characters in the conversation."""
        return sorted(list({
            line.character
            for page in self.pages
            for line in page.dialogue
        }))


conversations = []

curr_conv: list[ExtractedPage] = []

for page in extracted_pages:
    if curr_conv and (curr_conv[-1].page_num + 1) != page.page_num:
        conversations.append(Conversation(pages=curr_conv))
        curr_conv = []
    curr_conv.append(page)
if curr_conv:
    conversations.append(Conversation(pages=curr_conv))

len(conversations)

261

## Trim conversations

Remove all messages before Uno speaks for the first time and after he speaks last.

In [59]:
@dataclass
class DatasetDialogue:
    issue: str
    input_pages: list[str]
    dialogue: list[ExtractedLine]


def create_dataset(conversations: list[Conversation]) -> list[DatasetDialogue]:
    """Create dataset dialogues from conversations."""
    dialogues = []

    for conv in conversations:
        issue = conv.issue
        input_pages = [
            str(page.input_path) for page in conv.pages
        ]
        dialogue = [line for page in conv.pages for line in page.dialogue]

        if 'uno' not in [l.character for l in dialogue]:
            continue

        # Group together lines with the same character appearing consecutively
        curr_lines: list[ExtractedLine] = []
        grouped: list[list[ExtractedLine]] = []
        for line in dialogue:
            if curr_lines and curr_lines[-1].character != line.character:
                grouped.append(curr_lines)
                curr_lines = []
            curr_lines.append(line)
        if curr_lines:
            grouped.append(curr_lines)

        # Create line objects for each group
        dialogue = [
            ExtractedLine(
                character=group[0].character,
                text="\n".join([l.text for l in group]),
            )
            for group in grouped
        ]

        # Find the first and last index of lines with 'uno'
        first_uno = next((i for i, line in enumerate(dialogue) if line.character == 'uno'))
        last_uno = len(dialogue) - next((i for i, line in enumerate(reversed(dialogue)) if line.character == 'uno'))
        # Keep only the first line before 'uno' and his last reply.
        first = first_uno-1 if first_uno>0 else 0
        last = last_uno if last_uno<len(dialogue) else len(dialogue)
        dialogue = dialogue[first:last]

        dialogues.append(DatasetDialogue(
            issue=issue,
            input_pages=input_pages,
            dialogue=dialogue,
        ))

    return dialogues

dataset = create_dataset(conversations)
dataset[0]

DatasetDialogue(issue='PKNA #0', input_pages=['../input/pkna/pkna-0/pkna-0-029.jpg', '../input/pkna/pkna-0/pkna-0-030.jpg', '../input/pkna/pkna-0/pkna-0-031.jpg', '../input/pkna/pkna-0/pkna-0-032.jpg', '../input/pkna/pkna-0/pkna-0-033.jpg', '../input/pkna/pkna-0/pkna-0-034.jpg', '../input/pkna/pkna-0/pkna-0-035.jpg', '../input/pkna/pkna-0/pkna-0-036.jpg'], dialogue=[ExtractedLine(character='pk', text='Hai paura di affrontarmi? Fatti vedere!\nVeramente, mi stai già guardando!'), ExtractedLine(character='uno', text='Io sono questo edificio!'), ExtractedLine(character='pk', text='Sglurg!'), ExtractedLine(character='uno', text='Ma se proprio hai bisogno di un volto a cui rivolgerti ... ecco!'), ExtractedLine(character='pk', text='Glom! Penso di averne viste abbastanza, per una sola notte!\nFacciamoci coraggio!\nNon so cosa tu sia, ma hai di fronte Paperi-Nik!'), ExtractedLine(character='uno', text='Piacere! Io sono uno!'), ExtractedLine(character='pk', text='Uno, eh? Dove hai lasciato gli 

## Save dataset

In [60]:
import pandas as pd


def to_chat(dialogue: list[ExtractedLine]) -> list[dict[str, str]]:
    """Convert dialogue lines to chat format. Merge consecutive lines by the
    same character."""
    merged = []
    for line in dialogue:
        role = 'assistant' if line.character == 'uno' else 'human'
        # If the last line is from the same character, merge them
        # Otherwise, create a new entry
        if merged and merged[-1]['role'] == role:
            merged[-1]['content'] += "\n" + line.text
        else:
            merged.append({
                'role': role,
                'content': line.text
            })
    return merged


records = [
    {
        'issue': d.issue,
        'input_pages': d.input_pages,
        'characters': sorted(list(set(line.character for line in d.dialogue))),
        'conversations': to_chat(d.dialogue),
    }
    for d in dataset
]

df = pd.DataFrame.from_records(records)
df.head()

,issue,input_pages,characters,conversations
0,PKNA #0,"[../input/pkna/pkna-0/pkna-0-029.jpg, ../input...","[other, pk, uno]","[{'role': 'human', 'content': 'Hai paura di af..."
1,PKNA #0,"[../input/pkna/pkna-0/pkna-0-046.jpg, ../input...","[other, pk, uno]","[{'role': 'human', 'content': 'Ci sei, uno? Pu..."
2,PKNA #0,"[../input/pkna/pkna-0/pkna-0-057.jpg, ../input...","[other, pk, uno]","[{'role': 'human', 'content': 'Intervento prov..."
3,PKNA #0,"[../input/pkna/pkna-0/pkna-0-069.jpg, ../input...","[pk, uno]","[{'role': 'human', 'content': 'Sempre su... Eh..."
4,PKNA #0/2,"[../input/pkna/pkna-0-2/PK0-2 005.jpg, ../inpu...","[pk, uno]","[{'role': 'human', 'content': 'Buongiorno, uno..."


In [61]:
len(df)

244

In [62]:
df.to_csv(output_dataset, index=False)